# Musanga V4 — NB07 v4_007: faster resumable RGB+MS inference

Groups up to 4x4 native 256px output tiles per sensor read, batches up to eight 7-channel tensors on the GPU per fold, and checkpoints completed per-mosaic tiles every 250 outputs. Keeps the previous exact output filenames and QA/model fingerprints, so previously committed halo064 tiles are reused without rerunning inference. Three fold checkpoints, 64px halo, ROI setback, per-flight complete MS coverage and 100m interior remain unchanged. No model retraining.

Pilot with SITE=Baego and MAX_TILES=64; then SITE=ALL and MAX_TILES=None. Interrupt the old run before launching a new one; avoid two concurrent writers. Minor floating point differences between single-tile and batched forward passes are possible.

## Notebook workflow and status

- **Sections 00–04:** deterministic, resumable RGB + multispectral UAV inference using existing fold checkpoints; no retraining.
- **Section 05:** six-panel visual inspection with reference annotation outlines.
- **Sections 21–28:** post-inference inventory, integrity checks, georeferencing/coverage checks, visual QA and optional held-out validation. Outputs go to `Outputs/v4/07_post_inference_QA/`; paired validation requires explicitly supplied reference rasters and held-out masks.
- **Section 29:** reconstructs six georeferenced probability/binary mosaics from already-cached tiles, enforcing the model `836ab7d77c1c`, QA policy `47890f94c6cf`, 64-pixel halo, and expected tile counts. Reconstruction is resumable and writes to `Outputs/v4/07_RGB_MS_100m_ROI_complete_sensors_v4/reconstructed_mosaics/`.

**Interpretation:** Sections 21–29 define checks and outputs; their presence in GitHub does **not** establish that the corresponding Colab cells ran successfully or that all six mosaics were produced. Inspect saved manifests and output files before reporting completion. Existing cached prediction tiles are not modified by these sections.


## 00 — Input registry and reproducible objective inference settings

In [ ]:
%pip -q install rasterio geopandas pyogrio shapely scipy pandas matplotlib segmentation-models-pytorch
from pathlib import Path
from contextlib import ExitStack
from collections import defaultdict
import os,json,hashlib,gc,warnings,math
import numpy as np,pandas as pd,geopandas as gpd,matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window,from_bounds,transform as window_transform
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.transform import Affine
from rasterio.features import rasterize
from rasterio.warp import reproject
from scipy.ndimage import distance_transform_edt
from shapely.geometry import box
from shapely.ops import unary_union
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
ROOT=next((p for p in (Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                        Path('/content/drive/MyDrive/Musanga/Final')) if p.is_dir()),None)
assert ROOT is not None
DATA=ROOT/'Data';V4=ROOT/'Outputs/v4';AUDIT=V4/'00_input_audit'
OLD=ROOT/'Outputs/V3_1'; NB05=V4/'05_rgb_dsm_ms_matched_v1'
MODEL_DIR=NB05/'04_fusion_models'
OUT=V4/'07_RGB_MS_100m_ROI_complete_sensors_v4'
QA_DIR=OUT/'quality_maps'; TILE_DIR=OUT/'prediction_tiles';FIG=OUT/'figures'
for p in (QA_DIR,TILE_DIR,FIG):p.mkdir(parents=True,exist_ok=True)
V31=json.loads((OLD/'00_input_audit/00_input_manifest.json').read_text())
F78=json.loads((AUDIT/'00_v4_field78_contract.json').read_text())
TRAIN=json.loads((V4/'03_paired_rgb_unet_v1/03_training_contract.json').read_text())
REG=pd.read_csv(NB05/'05_harmonized_sensor_registry.csv')
assert {'group','band','path','data_band'}.issubset(REG.columns)
MOSAICS={}
for v in V31['mosaics']:
 if str(v['file']).endswith('.tif'):
  p=Path(v['path'])
  if not p.is_file():p=DATA/'Ortho_RGB'/v['file']
  assert p.is_file(),p
  MOSAICS[v['file']]=p
assert len(MOSAICS)==6
# The exact same-flight MS registry built by NB05 (including Yoko P1/P2).
GROUP_MOSAIC={
 'Baego_MS':'Baego.tif',
 'MOS_1_2_MS':'Mos_1_2_transparent_mosaic_group1_georef.tif',
 'MOS_3_4_MS':'Mos_3_4_transparent_mosaic_group1_georef.tif',
 'MOS_5_6_MS':'Mos_5_6_transparent_mosaic_group1_georef.tif',
 'Yangambi_YBJoseph_right_MS':'Yangambi_georef.tif',
 'Yangambi_YBForgeo_right_MS':'Yangambi_georef.tif'}
for group in REG.group.unique():
 if str(group).startswith('Yoko_Carlier2020_MicaSense_'):
  yoko=[m for m in MOSAICS if 'yoko' in m.lower()]
  assert len(yoko)==1,yoko
  GROUP_MOSAIC[group]=yoko[0]
assert set(REG.group.unique())==set(GROUP_MOSAIC),(
 'MS registry changed since NB05',set(REG.group.unique())^set(GROUP_MOSAIC))
ROI=gpd.read_file(Path(V31['roi'])).to_crs('EPSG:32635')
assert ROI.crs.is_projected
ROI_FULL=unary_union(ROI.geometry)
assert ROI_FULL.is_valid and not ROI_FULL.is_empty
SITE='ALL'   # 'Baego', 'MOS', 'Yangambi', 'Yoko', 'ALL'
ROI_SETBACK_M=100.0
ROI_INTERIOR=ROI_FULL.buffer(-ROI_SETBACK_M,join_style=1)
assert not ROI_INTERIOR.is_empty,'100 m inward buffer removed the complete ROI'
QA_RES_M=20.0   # coarse diagnostic only; inference uses native RGB pixels
MAX_TILES=None    # Reference-centred GPU pilot; None for all eligible tiles
EDGE_HALO_PX=64  # context beyond each 256px output tile; 64 makes 384px input
assert EDGE_HALO_PX>=0 and EDGE_HALO_PX%32==0
PRIORITIZE_REFERENCE_TILES=True
MAX_PRIORITY_FOOTPRINTS_PER_MOSAIC=12
QA_BLOCK=256    # maximum coarse QA-array dimensions per raster read
TILE=256
MIN_TILE_QA_FRACTION=0.01  # Efficiency only; not a quality threshold
PROB_THRESHOLD=.5
assert SITE in ('Baego','MOS','Yangambi','Yoko','ALL')
def site_of(m):
 return ('Baego' if m.startswith('Baego') else 'MOS' if m.startswith('Mos_')
         else 'Yangambi' if m.startswith('Yangambi') else 'Yoko')
SELECTED=[m for m in MOSAICS if SITE=='ALL' or site_of(m)==SITE]
def bands_for(group):
 q=REG[REG.group.eq(group)]
 assert len(q)==4 and set(q.band)=={'red','green','red_edge','nir'},group
 return {b:(Path(q[q.band.eq(b)].iloc[0].path),
            int(q[q.band.eq(b)].iloc[0].data_band))
         for b in ('red','green','red_edge','nir')}
for group in GROUP_MOSAIC:
 for p,_ in bands_for(group).values():assert p.is_file(),p
PAIRS=pd.read_csv(AUDIT/'00_rgb_dsm_pair_audit.csv')
DSM_BY_MOSAIC={Path(z.rgb_path).name:Path(z.dsm_path) for z in PAIRS.itertuples(index=False)}
assert set(MOSAICS)==set(DSM_BY_MOSAIC)
assert all(p.is_file() for p in DSM_BY_MOSAIC.values())
print('Mosaics:',SELECTED,'| ROI setback:',ROI_SETBACK_M,'m | outputs:',OUT)


## 01 — Fast source audit (headers ONLY; no resampling, no pixel reading)

The previous QA overview was unnecessary and could consume excessive RAM and minutes of I/O. This section now checks only raster metadata and records which RGB/DSM/MS flight sources exist. Sensor support is actually checked in Section 04 at the native RGB pixel scale.

In [ ]:
# No raster read() or WarpedVRT; only source headers and metadata.
# Keep a tiny registry, rather than constructing an enormous preview raster.
record=[]
for mosaic in SELECTED:
 with rasterio.open(MOSAICS[mosaic]) as rgb, rasterio.open(DSM_BY_MOSAIC[mosaic]) as dsm:
  assert rgb.crs is not None and dsm.crs is not None
  assert rgb.count>=3 and all(np.dtype(t)==np.dtype('uint8') for t in rgb.dtypes[:3])
  flights=[g for g,m in GROUP_MOSAIC.items() if m==mosaic]
  assert flights,('No complete MS source registered',mosaic)
  for group in flights:
   for band,(p,idx) in bands_for(group).items():
    with rasterio.open(p) as src:
     assert src.crs is not None and 1<=idx<=src.count,(group,band,p)
  record.append(dict(mosaic=mosaic,flight=';'.join(flights),
    qa_path='',roi_interior_pixels=np.nan,accepted_fraction=np.nan,
    rgb_resolution_m=abs(rgb.transform.a),
    rgb_width=rgb.width,rgb_height=rgb.height,
    dsm_resolution_m=abs(dsm.transform.a),n_four_band_MS_flights=len(flights)))
QA=pd.DataFrame(record)
QA.to_csv(OUT/'01_source_header_audit.csv',index=False)
display(QA[['mosaic','flight','rgb_resolution_m','n_four_band_MS_flights']])
print('Header checks done; no imagery loaded. Proceed to inference.')


## 02 — Optional: frozen reference inventory (no source-image reads)

We can revisit reference coverage once inference outputs exist. This inventory is not required to run the model.

In [ ]:
UNITS=gpd.read_file(V4/'01_spatial_cover_folds_v2/01_reference_groups_and_folds_v2.gpkg')
UNITS['unit_id']=UNITS.unit_id.astype(str)
OWNERS=pd.read_csv(V4/'02_rgb_circles_plus_plots_v3/02_reference_mosaic_owners_v2.csv')
OWNERS['unit_id']=OWNERS.unit_id.astype(str)
UNITS=UNITS.merge(OWNERS[['unit_type','unit_id','mosaic','roi_mosaic_coverage_fraction']],
 on=['unit_type','unit_id'],validate='one_to_one')
UNITS=UNITS[UNITS.roi_mosaic_coverage_fraction>=.95].copy()
assert len(UNITS)==106
display(UNITS[UNITS.mosaic.isin(SELECTED)].groupby(['site','unit_type'])
        .size().rename('footprints').reset_index())


## 03 — Deterministic inference readiness (no approval flag, no overview scan)

Every output pixel is checked against the actual ROI100, RGB, DSM and all four MS bands when its tile is processed.

In [ ]:
assert ROI_SETBACK_M==100.0
assert len(QA)==len(SELECTED)
assert all(QA.n_four_band_MS_flights>0)
print('Ready: strict 100 m interior ROI + RGB + DSM + four-band same-flight MS.')
print('Initial test:',SITE,'MAX_TILES:',MAX_TILES)


## 04 — Fast grouped sensor reads + batched GPU inference with existing tile reuse


In [ ]:
import torch
import segmentation_models_pytorch as smp
from rasterio.windows import bounds as window_bounds
assert torch.cuda.is_available(),'Select a GPU Colab runtime and rerun 00–03'
DEVICE='cuda'
def rgb_model_A(fold):
 nb03=V4/'03_paired_rgb_unet_v1'
 ck=torch.load(nb03/f'fold{fold}_A_{TRAIN["training_key"][:18]}.pt',
               map_location='cpu',weights_only=False)
 assert ck['training_key']==TRAIN['training_key']
 net=smp.Unet(encoder_name='resnet18',encoder_weights=None,in_channels=3,classes=1)
 net.load_state_dict({k.removeprefix('net.'):v for k,v in ck['model'].items()})
 return net
def ms_net(fold):
 base=rgb_model_A(fold).state_dict()
 net=smp.Unet(encoder_name='resnet18',encoder_weights=None,in_channels=7,classes=1)
 state=net.state_dict()
 for k,v in state.items():
  if k=='encoder.conv1.weight':
   v[:,:3]=base[k];v[:,3:]=0
  else:v.copy_(base[k])
 net.load_state_dict(state)
 ck=torch.load(MODEL_DIR/f'fold{fold}_RGB_MS_v1.pt',
               map_location='cpu',weights_only=False)
 assert ck['complete'] and ck['baseline_training_key']==TRAIN['training_key']
 net.load_state_dict(ck['state'],strict=True)
 return net.to(DEVICE).eval(),ck['fingerprint']
MODELS=[];MODEL_HASH=[]
for fold in range(3):
 model,h=ms_net(fold)
 MODELS.append(model);MODEL_HASH.append(str(h))
MODEL_SIGNATURE=hashlib.sha256(json.dumps(MODEL_HASH).encode()).hexdigest()[:12]
QA_SIGNATURE=hashlib.sha256(json.dumps(dict(
 schema='objective_ROI100_RGB_DSM_MS_v6_halo',setback=ROI_SETBACK_M,
 edge_halo_px=EDGE_HALO_PX,
 required=('RGB','DSM','red','green','red_edge','nir'),
 model=MODEL_SIGNATURE,registry=[
   (str(z.path),int(z.data_band),Path(z.path).stat().st_size,Path(z.path).stat().st_mtime_ns)
   for z in REG.itertuples()]),
 sort_keys=True).encode()).hexdigest()[:12]
print('NB05 RGB+MS model ensemble:',MODEL_SIGNATURE,'| coverage policy:',QA_SIGNATURE)
def make_vrts(ds,group,stack):
 result={}
 for b,(p,index) in bands_for(group).items():
  src=stack.enter_context(rasterio.open(p))
  vrt=stack.enter_context(WarpedVRT(src,crs=ds.crs,transform=ds.transform,
       width=ds.width,height=ds.height,resampling=Resampling.bilinear))
  result[b]=(vrt,index)
 return result
def read_ms_window(vrts,win):
 layers=[];valid=[]
 for b in ('red','green','red_edge','nir'):
  vrt,index=vrts[b]
  a=vrt.read(index,window=win,masked=True,out_dtype='float32')
  v=np.asarray(a.filled(0),np.float32)
  ok=(~np.ma.getmaskarray(a))&np.isfinite(v)&(v>0)
  layers.append(np.where(ok,v,0.));valid.append(ok)
 raw=np.stack(layers)
 valid=np.logical_and.reduce(valid)
 relative=np.where(valid[None],
        raw/np.maximum(raw.sum(axis=0,keepdims=True),1e-8),0.)
 return relative.astype('float32'),valid
def rgb_window(ds,win):
 a=ds.read([1,2,3],window=win,masked=True)
 assert all(np.dtype(t)==np.dtype('uint8') for t in ds.dtypes[:3])
 valid=(~np.ma.getmaskarray(a).any(axis=0))
 if ds.count>=4 and ds.colorinterp[3].name=='alpha':
  alpha=ds.read(4,window=win,masked=False)
  valid&=np.asarray(alpha)>0
 rgb=np.asarray(a.filled(0),np.float32)/255.
 return rgb,valid
def dsm_window(vrt,win):
 a=vrt.read(1,window=win,masked=True,out_dtype='float32')
 z=np.asarray(a.filled(np.nan),np.float32)
 return (~np.ma.getmaskarray(a))&np.isfinite(z)
def roi_window(ds,win,interior_geom):
 clip=interior_geom.intersection(box(*window_bounds(win,ds.transform)))
 if clip.is_empty:return np.zeros((int(win.height),int(win.width)),bool)
 return rasterize([(clip,1)],out_shape=(int(win.height),int(win.width)),
                  transform=ds.window_transform(win),fill=0,
                  dtype='uint8').astype(bool)
def tiles_for_roi(ds,interior):
 # Geometric tile discovery: no giant arrays, no missed narrow strips due
 # to a coarse sensor overview. Iterate only the inward ROI bounding range.
 if interior.is_empty:return []
 bb=rasterio.windows.from_bounds(*interior.bounds,transform=ds.transform)
 row0=max(0,int(np.floor(bb.row_off))//TILE)
 col0=max(0,int(np.floor(bb.col_off))//TILE)
 row1=min((ds.height-1)//TILE,int(np.ceil(bb.row_off+bb.height))//TILE)
 col1=min((ds.width-1)//TILE,int(np.ceil(bb.col_off+bb.width))//TILE)
 # Intersect ROI per tile, then check actual RGB/DSM/MS at full resolution
 # in run_mosaic. QA overview is not used for tile eligibility.
 return ((row,col) for row in range(row0,row1+1)
         for col in range(col0,col1+1)
         if interior.intersects(box(*rasterio.windows.bounds(
            Window(col*TILE,row*TILE,
              min(TILE,ds.width-col*TILE),min(TILE,ds.height-row*TILE)),
            ds.transform))))
def reference_centered_tiles(ds,interior,mosaic):
 # Choose complete test-footprint patches before exploring raster row order.
 # Reference selection does NOT use model output or measured Musanga cover.
 if not PRIORITIZE_REFERENCE_TILES or 'UNITS' not in globals():return []
 eligible=UNITS[UNITS.mosaic.eq(mosaic)].to_crs(ds.crs).copy()
 # favour circles and plots strictly inside eligible ROI and entirely on RGB
 eligible['interior_coverage']=[
   (g.intersection(interior).area/max(g.area,1e-9)) if g is not None and not g.is_empty else 0
   for g in eligible.geometry]
 eligible=eligible[eligible.interior_coverage>=.80].copy()
 eligible=eligible.sort_values(['unit_type','unit_id'],kind='stable')
 # Interleave plots and circles so each may appear in the atlas.
 groups=[eligible[eligible.unit_type.eq(k)] for k in ('circle','plot')]
 units=[]
 for records in zip(*[a.itertuples(index=False) for a in groups]):
  units.extend(records)
 # zip() may drop extra footprint types; append remaining sorted units.
 for row in eligible.itertuples(index=False):
  key=(row.unit_type,str(row.unit_id))
  if key not in {(q.unit_type,str(q.unit_id)) for q in units}:units.append(row)
 prioritized=[]
 included=0
 for z in units:
  if included>=MAX_PRIORITY_FOOTPRINTS_PER_MOSAIC:break
  # Slight halo: include full footprint bounding box and 1 tile margin
  bb=rasterio.windows.from_bounds(*z.geometry.bounds,transform=ds.transform)
  row0=max(0,int(np.floor(bb.row_off))//TILE-1)
  col0=max(0,int(np.floor(bb.col_off))//TILE-1)
  row1=min((ds.height-1)//TILE,int(np.ceil(bb.row_off+bb.height))//TILE+1)
  col1=min((ds.width-1)//TILE,int(np.ceil(bb.col_off+bb.width))//TILE+1)
  candidate=[(row,col) for row in range(row0,row1+1)
              for col in range(col0,col1+1)
              if interior.intersects(box(*rasterio.windows.bounds(
                Window(col*TILE,row*TILE,
                       min(TILE,ds.width-col*TILE),min(TILE,ds.height-row*TILE)),
                ds.transform)))]
  if not candidate:continue
  prioritized.extend(candidate)
  included+=1
  print('  Prioritizing',z.unit_type,z.unit_id,
        'ROI coverage',round(float(z.interior_coverage),2),
        '| tiles',len(candidate),flush=True)
 return list(dict.fromkeys(prioritized))
def core_and_context_window(ds,row,col):
 # Read only actual mosaic pixels; at edges zero-pad to a /32 input size.
 # Core georeferencing is ALWAYS determined from the original unpadded tile.
 r0=row*TILE;c0=col*TILE
 h=min(TILE,ds.height-r0);w=min(TILE,ds.width-c0)
 core=Window(c0,r0,w,h)
 context=Window(max(0,c0-EDGE_HALO_PX),max(0,r0-EDGE_HALO_PX),
    min(ds.width,c0+w+EDGE_HALO_PX)-max(0,c0-EDGE_HALO_PX),
    min(ds.height,r0+h+EDGE_HALO_PX)-max(0,r0-EDGE_HALO_PX))
 inner_r=int(r0-context.row_off);inner_c=int(c0-context.col_off)
 return core,context,(inner_r,inner_c,h,w)
def padded_context_features(rgb,ms):
 # Keep source alignment unchanged, pad only bottom/right to /32.
 h,w=rgb.shape[-2:]
 hh=int(np.ceil(h/32)*32);ww=int(np.ceil(w/32)*32)
 x=np.zeros((7,hh,ww),np.float32)
 x[:3,:h,:w]=rgb
 x[3:,:h,:w]=ms
 return x

# v4_007 fast/resumable engine: original model/quality signatures remain unchanged.
from collections import OrderedDict
from time import perf_counter
GPU_BATCH_TILES=8
READ_BLOCK_TILES=4
CHECKPOINT_EVERY=250
VERIFY_REUSED_TILES=False  # Old NB07 atomically renamed completed TIFFs.
assert GPU_BATCH_TILES>=1 and READ_BLOCK_TILES>=1

def output_path(folder,row,col):
 return folder/f'qa{QA_SIGNATURE}_m{MODEL_SIGNATURE}_halo{EDGE_HALO_PX:03d}_r{row:05d}_c{col:05d}.tif'

def grouped(coords):
 blocks=OrderedDict()
 for row,col in coords:
  blocks.setdefault((row//READ_BLOCK_TILES,col//READ_BLOCK_TILES),[]).append((row,col))
 return blocks.values()

def group_window(ds,jobs):
 contexts=[x[3] for x in jobs]
 x0=min(int(w.col_off) for w in contexts)
 y0=min(int(w.row_off) for w in contexts)
 x1=max(int(w.col_off+w.width) for w in contexts)
 y1=max(int(w.row_off+w.height) for w in contexts)
 return Window(x0,y0,x1-x0,y1-y0)

@torch.inference_mode()
def batch_predict(features):
 try:
  x=torch.from_numpy(np.stack(features)).to(DEVICE)
  acc=None
  for net in MODELS:
   with torch.autocast('cuda'):
    p=torch.sigmoid(net(x)[:,0]).float().cpu().numpy()
   if acc is None:acc=p.copy()
   else:acc+=p
  result=acc/len(MODELS)
  del x,acc
  return result
 except torch.cuda.OutOfMemoryError:
  torch.cuda.empty_cache()
  if len(features)==1:raise
  k=len(features)//2
  print('CUDA memory fallback: split batch of',len(features),flush=True)
  return np.concatenate([batch_predict(features[:k]),
                         batch_predict(features[k:])],axis=0)

def save_prediction(dest,rgb,core,prob,valid,flight):
 h,w=prob.shape
 binary=np.where(valid,(prob>=PROB_THRESHOLD).astype('float32'),-9999.)
 kw=dict(driver='GTiff',count=2,height=h,width=w,crs=rgb.crs,
         transform=rgb.window_transform(core),dtype='float32',
         nodata=-9999.,compress='deflate',tiled=False,interleave='band')
 tmp=dest.with_name(dest.stem+'.temp.tif')
 with rasterio.open(tmp,'w',**kw) as o:
  o.write(prob,1);o.write(binary,2)
  o.set_band_description(1,'Musanga probability, 64px contextual inference')
  o.set_band_description(2,'Musanga binary at probability threshold')
  o.update_tags(ms_flight=flight,roi_setback_m=ROI_SETBACK_M,
    edge_halo_px=EDGE_HALO_PX,model_folds='0,1,2',qa_signature=QA_SIGNATURE)
 os.replace(tmp,dest)

def checkpoint(mosaic,rows):
 path=OUT/('04_prediction_tile_manifest_'+mosaic.replace('.tif','')+'.csv')
 tmp=path.with_suffix('.partial.csv')
 pd.DataFrame(rows).to_csv(tmp,index=False)
 os.replace(tmp,path)

@torch.inference_mode()
def run_mosaic(mosaic):
 start=perf_counter()
 manifest=[]
 folder=TILE_DIR/mosaic.replace('.tif','')
 folder.mkdir(parents=True,exist_ok=True)
 globpat=f'qa{QA_SIGNATURE}_m{MODEL_SIGNATURE}_halo{EDGE_HALO_PX:03d}_r*_c*.tif'
 existing={p.name for p in folder.glob(globpat)}
 stats={'new':0,'reused':0,'skipped':0,'groups':0,'sensor_s':0.,'gpu_s':0.,'write_s':0.}
 last_checkpoint=0
 with ExitStack() as stack:
  rgb=stack.enter_context(rasterio.open(MOSAICS[mosaic]))
  dsm=stack.enter_context(rasterio.open(DSM_BY_MOSAIC[mosaic]))
  dsm_vrt=stack.enter_context(WarpedVRT(dsm,crs=rgb.crs,transform=rgb.transform,
                            width=rgb.width,height=rgb.height,resampling=Resampling.bilinear))
  interior=gpd.GeoSeries([ROI_INTERIOR],crs='EPSG:32635').to_crs(rgb.crs).iloc[0]
  flights={g:make_vrts(rgb,g,stack) for g,m in GROUP_MOSAIC.items() if m==mosaic}
  priority=reference_centered_tiles(rgb,interior,mosaic)
  priority_set=set(priority)
  def remaining():
   for coord in tiles_for_roi(rgb,interior):
    if coord not in priority_set:yield coord
  print(mosaic,'prioritized',len(priority),'existing',len(existing),
        'sensor block',READ_BLOCK_TILES,'GPU batch',GPU_BATCH_TILES,flush=True)
  for phase,coords in [('reference',priority),('remaining',remaining())]:
   for positions in grouped(coords):
    if MAX_TILES is not None and len(manifest)>=MAX_TILES:break
    to_do=[]
    for row,col in positions:
     if MAX_TILES is not None and len(manifest)+len(to_do)>=MAX_TILES:break
     core,ctx,inner=core_and_context_window(rgb,row,col)
     if not interior.intersects(box(*rasterio.windows.bounds(core,rgb.transform))):continue
     path=output_path(folder,row,col)
     if path.name in existing:
      okay=True
      if VERIFY_REUSED_TILES:
       try:
        with rasterio.open(path) as check:
         okay=(check.count==2 and check.width==int(core.width) and
               check.height==int(core.height) and check.nodata==-9999. and
               check.tags().get('qa_signature')==QA_SIGNATURE and
               check.tags().get('edge_halo_px')==str(EDGE_HALO_PX))
       except Exception:okay=False
      if okay:
       manifest.append(dict(mosaic=mosaic,row=row,col=col,path=str(path),
                            status='reused',sensor_flight='cached'))
       stats['reused']+=1;continue
      print('Damaged cached tile; regenerate',path.name,flush=True)
      existing.remove(path.name)
     to_do.append((row,col,core,ctx,inner,path))
    if not to_do:
     if len(manifest)-last_checkpoint>=CHECKPOINT_EVERY:
      checkpoint(mosaic,manifest);last_checkpoint=len(manifest)
     continue
    stats['groups']+=1
    tick=perf_counter()
    win=group_window(rgb,to_do)
    rgb_block,rgb_ok=rgb_window(rgb,win)
    dsm_ok=dsm_window(dsm_vrt,win)
    basic_jobs=[]
    for row,col,core,ctx,inner,path in to_do:
     ro=int(core.row_off-win.row_off);co=int(core.col_off-win.col_off)
     h,w=int(core.height),int(core.width)
     roi=roi_window(rgb,core,interior)
     basic=roi & rgb_ok[ro:ro+h,co:co+w] & dsm_ok[ro:ro+h,co:co+w]
     if basic.any():basic_jobs.append((row,col,core,ctx,inner,path,basic))
     else:stats['skipped']+=1
    if not basic_jobs:
     stats['sensor_s']+=perf_counter()-tick
     continue
    # Each four-band flight is reprojected once per spatial block.
    ms_blocks={g:read_ms_window(vrts,win) for g,vrts in flights.items()}
    tasks=[]
    for row,col,core,ctx,inner,path,basic in basic_jobs:
     ro=int(core.row_off-win.row_off);co=int(core.col_off-win.col_off)
     h,w=int(core.height),int(core.width)
     choices=[(int(np.count_nonzero(basic & valid[ro:ro+h,co:co+w])),g)
              for g,(ms,valid) in ms_blocks.items()]
     count,flight=max(choices)
     ms,validms=ms_blocks[flight]
     valid=basic & validms[ro:ro+h,co:co+w]
     if count==0 or valid.mean()<MIN_TILE_QA_FRACTION:
      stats['skipped']+=1;continue
     cr=int(ctx.row_off-win.row_off);cc=int(ctx.col_off-win.col_off)
     hh,ww=int(ctx.height),int(ctx.width)
     feat=padded_context_features(rgb_block[:,cr:cr+hh,cc:cc+ww],
                                  ms[:,cr:cr+hh,cc:cc+ww])
     tasks.append((row,col,core,inner,path,flight,valid,feat))
    stats['sensor_s']+=perf_counter()-tick
    del ms_blocks,rgb_block,rgb_ok,dsm_ok
    buckets=OrderedDict()
    for t in tasks:buckets.setdefault(t[7].shape,[]).append(t)
    for shape,items in buckets.items():
     for i in range(0,len(items),GPU_BATCH_TILES):
      batch=items[i:i+GPU_BATCH_TILES]
      tick=perf_counter()
      p=batch_predict([x[7] for x in batch])
      torch.cuda.synchronize()
      stats['gpu_s']+=perf_counter()-tick
      tick=perf_counter()
      for k,(row,col,core,inner,path,flight,valid,feat) in enumerate(batch):
       ir,ic,h,w=inner
       pred=p[k,ir:ir+h,ic:ic+w].copy().astype('float32')
       pred[~valid]=-9999.
       save_prediction(path,rgb,core,pred,valid,flight)
       manifest.append(dict(mosaic=mosaic,row=row,col=col,path=str(path),
                 status='new',sensor_flight=flight,valid_fraction=float(valid.mean())))
       stats['new']+=1
      stats['write_s']+=perf_counter()-tick
      del p
    if len(manifest)-last_checkpoint>=CHECKPOINT_EVERY:
     checkpoint(mosaic,manifest);last_checkpoint=len(manifest)
     seconds=perf_counter()-start
     print(mosaic,'completed',len(manifest),'new',stats['new'],'reused',stats['reused'],
       'skipped',stats['skipped'],'tiles/s',round(len(manifest)/max(seconds,1),2),
       'read',round(stats['sensor_s']),'GPU',round(stats['gpu_s']),
       'write',round(stats['write_s']),flush=True)
   if MAX_TILES is not None and len(manifest)>=MAX_TILES:break
  checkpoint(mosaic,manifest)
 print('FINISHED',mosaic,len(manifest),'tiles',
       'elapsed_min',round((perf_counter()-start)/60,1),'stats',stats,flush=True)
 return manifest

TILE_MANIFEST=[]
for mosaic in SELECTED:TILE_MANIFEST.extend(run_mosaic(mosaic))
TILE_MANIFEST=pd.DataFrame(TILE_MANIFEST)
TILE_MANIFEST.to_csv(OUT/'04_prediction_tile_manifest.csv',index=False)
print('Inference tiles saved/reused:',len(TILE_MANIFEST))
if len(TILE_MANIFEST):
 display(TILE_MANIFEST.groupby(['mosaic','status']).size().rename('tiles').reset_index())
else:print('No accepted inference tiles. Review ROI and sensor support.')


## 05 — Six-panel atlas with high-visibility reference annotation outlines

White, black-cased **individual crown annotation boundaries** are overlaid on panels 3–6, including the probability, binary and red-on-RGB predictions. The yellow line is **the reference footprint boundary**, not an individual crown. Predictions and masks are unchanged by plotting. The atlas only uses the current halo prediction tile manifest (not old uncropped files).

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from rasterio.merge import merge
# To keep figures honest, use only units actually intersecting saved prediction
# tiles and require >=50% quality-approved support within the shown unit.
assert 'TILE_MANIFEST' in globals() and len(TILE_MANIFEST)>0, 'Run Section 04 first. No accepted inference tiles exist.'
PLOT_SOURCE=gpd.read_file(F78['exhaustive_plot_crowns']).to_crs('EPSG:32635')
PLOT_SOURCE['positive']=pd.to_numeric(PLOT_SOURCE.Class_ID,errors='coerce').eq(1)
CIRCLE_SOURCE=gpd.read_file(F78['circle_positive_crowns']).to_crs('EPSG:32635')
CIRCLE_SOURCE['positive']=True
def plot_a_few():
 used=[]
 for mosaic,files in TILE_MANIFEST.groupby('mosaic'):
  units=UNITS[UNITS.mosaic.eq(mosaic)].copy()
  if not len(units):continue
  tilefiles=list(files.path)
  # Reference footprints can straddle tile boundaries. Select one footprint
  # whose whole buffered bounding box intersects saved tiles; mosaic merge
  # below reconstructs an excerpt from the georeferenced probability tiles.
  with ExitStack() as stack:
   src=[stack.enter_context(rasterio.open(p)) for p in tilefiles]
   union_bounds=unary_union([box(*p.bounds) for p in src])
   with rasterio.open(MOSAICS[mosaic]) as rgb:
    with ExitStack() as vst:
     group=next((g for g,m in GROUP_MOSAIC.items() if m==mosaic),None)
     vrts=make_vrts(rgb,group,vst)
     for z in units.to_crs(rgb.crs).itertuples():
      if not union_bounds.intersects(z.geometry):continue
      win=from_bounds(*z.geometry.bounds,transform=rgb.transform)
      c0=max(0,int(np.floor(win.col_off)));r0=max(0,int(np.floor(win.row_off)))
      c1=min(rgb.width,int(np.ceil(win.col_off+win.width)))
      r1=min(rgb.height,int(np.ceil(win.row_off+win.height)))
      ww=c1-c0;hh=r1-r0
      if ww<15 or hh<15 or ww*hh>8_000_000:continue
      window=Window(c0,r0,ww,hh)
      ptiles=[q for q in src if box(*q.bounds).intersects(z.geometry)]
      if not ptiles:continue
      pred,tr=merge(ptiles,bounds=rasterio.windows.bounds(window,rgb.transform),
                    nodata=-9999.,indexes=[1],res=(abs(rgb.transform.a),abs(rgb.transform.e)))
      p=np.asarray(pred[0],float)
      if p.shape!=(hh,ww):continue
      refmask=rasterize([(z.geometry,1)],out_shape=(hh,ww),
               transform=rgb.window_transform(window),fill=0).astype(bool)
      good=(p>=0)&(p<=1)
      if not refmask.any() or good[refmask].mean()<.20:continue
      rgb3,rv=rgb_window(rgb,window)
      ms,msv=read_ms_window(vrts,window)
      false=np.stack([ms[3],ms[2],ms[0]],axis=-1)
      validMS=msv&(ms.sum(axis=0)>0)
      if validMS.any():
       for band in range(3):
        q=false[:,:,band]
        lo,hi=np.percentile(q[validMS],[2,98])
        false[:,:,band]=np.clip((q-lo)/max(hi-lo,1e-8),0,1)
      false[~validMS]=1
      extent=rasterio.windows.bounds(window,rgb.transform)
      extent=[extent[0],extent[2],extent[1],extent[3]]
      fig,axes=plt.subplots(1,6,figsize=(24,4.2),layout='constrained')
      axes[0].imshow(np.moveaxis(rgb3,0,-1),extent=extent,origin='upper')
      axes[1].imshow(false,extent=extent,origin='upper')
      axes[2].imshow(np.moveaxis(rgb3,0,-1),extent=extent,origin='upper')
      src_gdf=CIRCLE_SOURCE if z.unit_type=='circle' else PLOT_SOURCE
      geom32635=gpd.GeoSeries([z.geometry],crs=rgb.crs).to_crs('EPSG:32635').iloc[0]
      selected=src_gdf.iloc[src_gdf.sindex.query(geom32635,predicate='intersects')].copy()
      if len(selected):
       selected=selected.to_crs(rgb.crs)
      probability=np.ma.masked_where(~good,p)
      axes[3].imshow(probability,extent=extent,origin='upper',
                     cmap='magma',vmin=0,vmax=1,interpolation='nearest')
      binary=np.ma.masked_where(~good,(p>=PROB_THRESHOLD).astype(float))
      axes[4].imshow(binary,extent=extent,origin='upper',
                     cmap='Greens',vmin=0,vmax=1,interpolation='nearest')
      axes[5].imshow(np.moveaxis(rgb3,0,-1),extent=extent,origin='upper')
      # Saturated red prediction is intentionally stronger than old spring overlay.
      predmask=np.ma.masked_where(~good|(p<PROB_THRESHOLD),np.ones_like(p))
      redmap=mcolors.ListedColormap(['#ff2735'])
      axes[5].imshow(predmask,extent=extent,origin='upper',
                     cmap=redmap,vmin=0,vmax=1,alpha=.60,interpolation='nearest')
      # Actual reference crowns, NOT the footprint bounding mask:
      # thick white lines with thin dark casing for contrast on all backgrounds.
      for ax in (axes[2],axes[3],axes[4],axes[5]):
       if len(selected):
        for polygon in selected.geometry:
         if polygon is None or polygon.is_empty:continue
         # GeoPandas border artists apply to Polygon and MultiPolygon.
         group=gpd.GeoSeries([polygon],crs=rgb.crs)
         before=len(ax.collections)
         group.boundary.plot(ax=ax,color='white',linewidth=1.65,zorder=15)
         for artist in ax.collections[before:]:
          artist.set_path_effects([pe.Stroke(linewidth=2.8,foreground='#101010'),
                                   pe.Normal()])
      for ax,title in zip(axes,['RGB','MS NIR/RE/R','Annotations','Probability',
                                 'Binary Musanga','Prediction overlay']):
       gpd.GeoSeries([z.geometry],crs=rgb.crs).boundary.plot(
          ax=ax,color='#ffec4d',linewidth=1.4)
       ax.set(xlim=(extent[0],extent[1]),ylim=(extent[2],extent[3]),title=title)
       ax.set_xticks([]);ax.set_yticks([])
      fig.suptitle(f'{z.site} {z.unit_type} {z.unit_id} | ROI100 + all sensors; valid={good[refmask].mean():.0%} | halo={EDGE_HALO_PX}px')
      out=FIG/f'example_halo{EDGE_HALO_PX:03d}_{mosaic.replace(".tif","")}_{z.unit_type}_{z.unit_id}.png'
      fig.savefig(out,dpi=175,bbox_inches='tight');plt.show();plt.close(fig)
      used.append(dict(site=z.site,unit_type=z.unit_type,unit_id=z.unit_id,
                       path=str(out),approved_fraction=float(good[refmask].mean())))
      if len(used)>=6:return used
 return used
EXAMPLES=pd.DataFrame(plot_a_few())
EXAMPLES.to_csv(OUT/'05_example_atlas_manifest.csv',index=False)
print('Completed QA example panels:',len(EXAMPLES))
display(EXAMPLES)
if not len(EXAMPLES):
 print('No reference footprint passed the strict overlap/ROI/sensor tests. '
       'Displaying genuine saved inference tiles instead (not validation footprints).')
 fig,axs=plt.subplots(1,3,figsize=(12,4),layout='constrained')
 shown=0
 for q in TILE_MANIFEST.itertuples(index=False):
  with rasterio.open(q.path) as src:
   p=src.read(1,masked=True)
   valid=~np.ma.getmaskarray(p)
   if not np.any(valid):continue
   axs[shown].imshow(p,cmap='magma',vmin=0,vmax=1)
   axs[shown].set_title(f'{q.mosaic} tile r{q.row} c{q.col}\nvalid={valid.mean():.0%}')
   axs[shown].axis('off')
   shown+=1
   if shown==3:break
 for ax in axs[shown:]:ax.axis('off')
 if shown:
  fig.savefig(FIG/f'05_fallback_prediction_tile_preview_halo{EDGE_HALO_PX:03d}.png',
              dpi=160,bbox_inches='tight')
  plt.show()
 plt.close(fig)


In [ ]:

# ================================================================
# MUSANGA V4 — NB07 POST-INFERENCE QA AND VALIDATION
# ================================================================
# Append after completed NB07 inference.
#
# 1. Inventory saved tiles across all six mosaics
# 2. Check tile integrity and coverage
# 3. Identify georeferenced predictions / mosaic readiness
# 4. Generate summary plots
# 5. Compare with independent reference rasters when available
#
# READ-ONLY for original inference products.
# No GPU, no retraining, no deletion.
#
# Output: Final/Outputs/v4/07_post_inference_QA/
# ================================================================

from pathlib import Path
from collections import Counter
import json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import rasterio
    from rasterio.windows import Window
except ImportError:
    %pip -q install rasterio
    import rasterio
    from rasterio.windows import Window

from google.colab import drive as colab_drive
colab_drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/Musanga/Final"
)
V4 = ROOT / "Outputs" / "v4"

INFERENCE = (
    V4 / "07_RGB_MS_100m_ROI_complete_sensors_v4"
)
TILES = INFERENCE / "prediction_tiles"
QUALITY = INFERENCE / "quality_maps"
FIGURES = INFERENCE / "figures"

# Separate QA directory; original inference remains untouched
OUT = V4 / "07_post_inference_QA"
OUT.mkdir(parents=True, exist_ok=True)

EXPECTED = {
    "Baego": 1568,
    "Mos_1_2_transparent_mosaic_group1_georef": 18464,
    "Mos_3_4_transparent_mosaic_group1_georef": 13816,
    "Mos_5_6_transparent_mosaic_group1_georef": 6582,
    "Yangambi_georef": 4552,
    "Yoko_kvo_georef": 3272,
}

EXPECTED_MODEL = "836ab7d77c1c"
EXPECTED_POLICY = "47890f94c6cf"

assert TILES.exists(), f"Missing tile directory: {TILES}"

print("=" * 78)
print("NB07 POST-INFERENCE QA")
print("Inference directory:", INFERENCE)
print("Expected model:", EXPECTED_MODEL)
print("Expected coverage policy:", EXPECTED_POLICY)
print("=" * 78)

# ================================================================
# SECTION 21 — TILE INVENTORY
# ================================================================

# Recognize common saved tile formats.
# The extension is not sufficient to prove that an object
# is a valid prediction. We inspect sample files below.

TILE_EXT = {".tif", ".tiff", ".npz", ".npy"}

rows = []
tile_files = {}

for mosaic, expected in EXPECTED.items():

    folder = TILES / mosaic

    if not folder.exists():
        print("[MISSING DIRECTORY]", mosaic)
        files = []
    else:
        files = sorted(
            p for p in folder.rglob("*")
            if p.is_file() and p.suffix.lower() in TILE_EXT
        )

    tile_files[mosaic] = files

    extensions = Counter(p.suffix.lower() for p in files)

    rows.append({
        "mosaic": mosaic,
        "expected_tiles": expected,
        "files_found": len(files),
        "difference": len(files) - expected,
        "file_count_matches": len(files) == expected,
        "extensions": str(dict(extensions)),
        "folder_exists": folder.exists(),
    })

inventory = pd.DataFrame(rows)
inventory.to_csv(
    OUT / "21_tile_inventory.csv", index=False
)

display(inventory)

print("\nTotal files found:", inventory.files_found.sum())
print("Total expected:", sum(EXPECTED.values()))

if inventory.file_count_matches.all():
    print("[PASS] File counts match all six inference summaries.")
else:
    print(
        "[REVIEW] Some counts differ. This may reflect naming, "
        "formats or auxiliary files rather than missing predictions."
    )

# ================================================================
# SECTION 22 — TILE SCHEMA AND INTEGRITY SAMPLE
# ================================================================

# Inspect a deterministic sample from each mosaic.
# Avoid reading all 48,254 tiles from Google Drive.

SCHEMA_SAMPLE = 5

def describe_tile(path):

    result = {
        "filename": path.name,
        "extension": path.suffix.lower(),
        "readable": False,
        "shape": "",
        "dtype": "",
        "crs": "",
        "georeferenced": False,
        "keys": "",
        "error": "",
    }

    try:
        if path.suffix.lower() in {".tif", ".tiff"}:
            with rasterio.open(path) as ds:
                result["shape"] = str(
                    (ds.count, ds.height, ds.width)
                )
                result["dtype"] = str(ds.dtypes)
                result["crs"] = str(ds.crs)
                result["georeferenced"] = bool(
                    ds.crs is not None and
                    not ds.transform.is_identity
                )
                result["readable"] = True

        elif path.suffix.lower() == ".npz":
            with np.load(path, allow_pickle=False) as data:
                result["keys"] = ",".join(data.files)

                shapes = {
                    k: tuple(data[k].shape)
                    for k in data.files
                }

                result["shape"] = str(shapes)
                result["readable"] = True

        elif path.suffix.lower() == ".npy":
            arr = np.load(
                path, allow_pickle=False,
                mmap_mode="r"
            )
            result["shape"] = str(arr.shape)
            result["dtype"] = str(arr.dtype)
            result["readable"] = True

    except Exception as exc:
        result["error"] = str(exc)[:250]

    return result


schema_rows = []

for mosaic, files in tile_files.items():
    for path in files[:SCHEMA_SAMPLE]:
        info = describe_tile(path)
        info["mosaic"] = mosaic
        schema_rows.append(info)

schema = pd.DataFrame(schema_rows)

schema.to_csv(
    OUT / "22_tile_schema_sample.csv",
    index=False
)

display(schema)

print("\nReadable sampled tiles:",
      int(schema.readable.sum()), "/", len(schema))

# ================================================================
# SECTION 23 — OUTPUT COVERAGE AND RECONSTRUCTION READINESS
# ================================================================

# We must not infer geographic tile positions from filenames
# without checking the actual NB07 tile schema.
#
# GeoTIFF tile collections can be assembled by georeferencing.
# NPZ/NPY tiles require explicit window positions, CRS and source
# raster metadata. Those must be obtained from saved manifests
# or notebook variables, never guessed.

geo_rows = []

for mosaic, files in tile_files.items():

    tif_files = [
        p for p in files
        if p.suffix.lower() in {".tif", ".tiff"}
    ]

    valid_geo = 0
    crs_values = set()

    for path in tif_files[:10]:
        try:
            with rasterio.open(path) as src:
                if src.crs and not src.transform.is_identity:
                    valid_geo += 1
                    crs_values.add(str(src.crs))
        except Exception:
            pass

    geo_rows.append({
        "mosaic": mosaic,
        "tif_tile_count": len(tif_files),
        "georeferenced_in_sample": valid_geo,
        "sampled_tifs": min(10, len(tif_files)),
        "sample_crs": str(sorted(crs_values)),
        "automatic_mosaic_ready": (
            len(tif_files) > 0
            and valid_geo == min(10, len(tif_files))
            and len(crs_values) == 1
        ),
    })

geo = pd.DataFrame(geo_rows)
geo.to_csv(
    OUT / "23_mosaic_readiness.csv", index=False
)

display(geo)

print("\nIMPORTANT:")
print("Readiness is a sample-based diagnostic.")
print("It is not proof that all tiles form a seamless mosaic.")
print("Full mosaic assembly requires checking tile overlap,")
print("valid-data masks, pixel alignment and blending policy.")

# ================================================================
# SECTION 24 — SITE COMPLETENESS FIGURES
# ================================================================

fig, axes = plt.subplots(
    1, 2, figsize=(14, 5),
    constrained_layout=True
)

labels = [
    "Baego", "MOS 1-2", "MOS 3-4",
    "MOS 5-6", "Yangambi", "Yoko"
]

x = np.arange(len(labels))

axes[0].bar(
    x, inventory.files_found.to_numpy()
)
axes[0].plot(
    x, inventory.expected_tiles.to_numpy(),
    "o", label="Expected"
)
axes[0].set_xticks(x, labels, rotation=35, ha="right")
axes[0].set_ylabel("Saved tile files")
axes[0].set_title("NB07 inference inventory")
axes[0].legend()

pct = (
    100 * inventory.files_found
    / inventory.expected_tiles
)

axes[1].bar(x, pct)
axes[1].axhline(100, ls="--", color="black")
axes[1].set_xticks(x, labels, rotation=35, ha="right")
axes[1].set_ylabel("Files / expected tiles (%)")
axes[1].set_title("Tile-count completeness")

fig.savefig(
    OUT / "24_tile_completeness.png",
    dpi=250, bbox_inches="tight"
)
plt.show()

# ================================================================
# SECTION 25 — REFERENCE-FOOTPRINT COVERAGE AUDIT
# ================================================================

# NB07 already generated six example panels. Search for the
# saved coverage tables without modifying them.
#
# A table must explicitly contain the unit, site and coverage
# fields to be treated as reference coverage information.

coverage_tables = []

SEARCH_DIRS = [
    INFERENCE,
    V4 / "07_RGB_MS_quality_gated_inference_v1",
]

for base in SEARCH_DIRS:
    if not base.exists():
        continue

    # Limit discovery to CSV files outside the large tile tree
    for p in base.glob("*.csv"):
        coverage_tables.append(p)
    for sub in ["figures", "quality_maps", "tables", "qa"]:
        d = base / sub
        if d.exists():
            coverage_tables.extend(d.glob("*.csv"))

coverage_frames = []

for path in sorted(set(coverage_tables)):
    try:
        df = pd.read_csv(path)
        cols = set(df.columns)

        if {"site", "unit_type", "unit_id",
            "approved_fraction"}.issubset(cols):

            z = df[[
                "site", "unit_type",
                "unit_id", "approved_fraction"
            ]].copy()

            z["source"] = path.name
            coverage_frames.append(z)
    except Exception:
        pass

if coverage_frames:

    coverage = pd.concat(
        coverage_frames, ignore_index=True
    ).drop_duplicates(
        ["site", "unit_type", "unit_id"],
        keep="last"
    )

    coverage.to_csv(
        OUT / "25_reference_coverage.csv",
        index=False
    )

    summary = coverage.groupby("site").agg(
        n_units=("unit_id", "count"),
        mean_coverage=("approved_fraction", "mean"),
        min_coverage=("approved_fraction", "min"),
        n_below_90=(
            "approved_fraction",
            lambda x: int((x < 0.90).sum())
        ),
    ).reset_index()

    display(summary)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(
        summary.site,
        summary.mean_coverage * 100
    )
    ax.axhline(90, ls="--", color="black")
    ax.set_ylim(0, 105)
    ax.set_ylabel("Mean approved coverage (%)")
    ax.set_title("Reference-footprint sensor coverage")
    fig.savefig(
        OUT / "25_reference_coverage.png",
        dpi=250, bbox_inches="tight"
    )
    plt.show()

else:
    print(
        "[NOT YET VERIFIED] No compatible coverage CSV found."
    )
    print(
        "The six example panels alone are insufficient "
        "to establish coverage for every forest."
    )

# ================================================================
# SECTION 26 — EXISTING RECONSTRUCTED MOSAIC INVENTORY
# ================================================================

# Only inspect existing GeoTIFF mosaics.
# Do not assemble tens of thousands of tiles blindly or
# overwrite model predictions.

mosaic_candidates = []

for base in [
    QUALITY, FIGURES,
    INFERENCE / "mosaics",
    INFERENCE / "predictions",
]:
    if not base.exists():
        continue

    for p in base.rglob("*.tif"):
        mosaic_candidates.append(p)
    for p in base.rglob("*.tiff"):
        mosaic_candidates.append(p)

mosaic_info = []

for path in mosaic_candidates:

    try:
        with rasterio.open(path) as ds:

            mosaic_info.append({
                "file": str(path),
                "width": ds.width,
                "height": ds.height,
                "bands": ds.count,
                "crs": str(ds.crs),
                "res_x": abs(ds.transform.a),
                "res_y": abs(ds.transform.e),
                "size_gb": path.stat().st_size / 1e9,
            })

    except Exception as exc:
        print("[UNREADABLE]", path.name, str(exc)[:120])

mosaic_df = pd.DataFrame(mosaic_info)

mosaic_df.to_csv(
    OUT / "26_existing_mosaics.csv", index=False
)

if len(mosaic_df):
    display(mosaic_df)
else:
    print(
        "No existing reconstructed GeoTIFF mosaics "
        "found in the checked NB07 output locations."
    )

# ================================================================
# SECTION 27 — PAIRED RASTER VALIDATION (OPTIONAL)
# ================================================================

# To compare new V4 predictions with V3_1 or independent
# annotations, provide explicit georeferenced raster paths.
#
# IMPORTANT:
# - Reference rasters must contain Musanga fractional cover
#   on a known scale (0–1 or 0–100).
# - Prediction rasters must represent the SAME quantity.
# - This validation never treats the same training pixels as
#   independent held-out evidence.
#
# Add the paths after verified mosaic reconstruction.
# The list is empty by default to prevent misleading results.

PAIRED_RASTERS = [
    # Example format:
    # {
    #     "site": "Baego",
    #     "prediction": "/content/drive/.../v4_prediction.tif",
    #     "reference": "/content/drive/.../reference_cover.tif",
    #     "heldout_mask": "/content/drive/.../heldout_mask.tif",
    # }
]

from rasterio.warp import reproject, Resampling

paired_results = []

for item in PAIRED_RASTERS:

    site = item["site"]

    with rasterio.open(item["prediction"]) as pred, \
         rasterio.open(item["reference"]) as ref, \
         rasterio.open(item["heldout_mask"]) as mask:

        # Read reference as destination grid.
        # For large mosaics, replace with windowed reading.
        if ref.width * ref.height > 25_000_000:
            print(
                "[SKIP]", site,
                "Raster too large for this in-memory validation."
            )
            continue

        if not (pred.crs and ref.crs and mask.crs):
            raise ValueError(
                f"{site}: missing CRS in prediction/reference/mask"
            )

        y = ref.read(1, masked=True).filled(np.nan)

        p = np.full(
            y.shape, np.nan, dtype=np.float32
        )

        reproject(
            rasterio.band(pred, 1),
            p,
            src_transform=pred.transform,
            src_crs=pred.crs,
            src_nodata=pred.nodata,
            dst_transform=ref.transform,
            dst_crs=ref.crs,
            dst_nodata=np.nan,
            resampling=Resampling.average,
        )

        held = np.zeros(y.shape, dtype=np.uint8)

        reproject(
            rasterio.band(mask, 1),
            held,
            src_transform=mask.transform,
            src_crs=mask.crs,
            src_nodata=mask.nodata,
            dst_transform=ref.transform,
            dst_crs=ref.crs,
            dst_nodata=0,
            resampling=Resampling.nearest,
        )

        valid = (
            (held == 1)
            & np.isfinite(y)
            & np.isfinite(p)
        )

        if not valid.any():
            print("[SKIP]", site, "No valid held-out overlap.")
            continue

        obs = y[valid].astype(float)
        est = p[valid].astype(float)

        # Explicitly require matching fractional-cover units.
        if (obs.min() < 0 or obs.max() > 1 or
            est.min() < 0 or est.max() > 1):
            raise ValueError(
                f"{site}: expected 0–1 fractional cover rasters."
            )

        residual = 100 * (est - obs)

        mae = float(np.mean(np.abs(residual)))
        rmse = float(np.sqrt(np.mean(residual**2)))
        bias = float(np.mean(residual))

        observed_var = np.sum((obs - obs.mean())**2)
        r2 = (
            1 - np.sum((est - obs)**2) / observed_var
            if observed_var > 0 else np.nan
        )

        paired_results.append({
            "site": site,
            "n_pixels": int(valid.sum()),
            "MAE_pp": mae,
            "RMSE_pp": rmse,
            "bias_pp": bias,
            "R2": float(r2),
        })

if paired_results:

    paired_df = pd.DataFrame(paired_results)
    display(paired_df)

    paired_df.to_csv(
        OUT / "27_paired_raster_validation.csv",
        index=False
    )

else:
    print(
        "Paired validation not executed: no verified "
        "prediction/reference/mask triplets supplied."
    )

# ================================================================
# SECTION 28 — FINAL QA MANIFEST
# ================================================================

manifest = {
    "notebook": "NB07",
    "stage": "post_inference_QA",
    "model_hash_expected": EXPECTED_MODEL,
    "coverage_policy_expected": EXPECTED_POLICY,
    "expected_tiles": sum(EXPECTED.values()),
    "files_found": int(inventory.files_found.sum()),
    "count_check_passed": bool(
        inventory.file_count_matches.all()
    ),
    "sampled_tiles_readable": int(
        schema.readable.sum()
    ) if len(schema) else 0,
    "sampled_tiles_total": len(schema),
    "mosaics_discovered": len(mosaic_df),
    "paired_validation_completed": bool(paired_results),
    "source_inference_dir": str(INFERENCE),
    "qa_output_dir": str(OUT),
    "original_predictions_modified": False,
}

with open(OUT / "28_QA_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\n" + "=" * 78)
print("NB07 POST-INFERENCE QA SUMMARY")
print("=" * 78)

for k, v in manifest.items():
    print(f"{k}: {v}")

print("\nSaved QA outputs:", OUT)
print("Existing model predictions were not modified.")
print("=" * 78)


## 28b — Optional I/O diagnostic (16 tiles only)

Run this **instead of Section 29** to benchmark Google Drive tile reads versus local reads. CPU-only, does not modify outputs.


In [ ]:
# NB07 I/O DIAGNOSTIC — SMALL SAMPLE, no mosaic construction, no Drive writes.
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import rasterio, shutil, tempfile, time, numpy as np
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
base=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/v4/07_RGB_MS_100m_ROI_complete_sensors_v4')
folder=base/'prediction_tiles/Mos_1_2_transparent_mosaic_group1_georef'
files=sorted(folder.glob('qa47890f94c6cf_m836ab7d77c1c_halo064_*.tif'))
assert len(files)==18464, f'Expected 18,464 tiles, found {len(files)}'
sample=[files[int(i)] for i in np.linspace(0,len(files)-1,16)]
def read_one(p):
 with rasterio.open(p) as ds:
  return float(ds.read(1).sum())
def bench(label,inputs,workers):
 t=time.perf_counter()
 with ThreadPoolExecutor(max_workers=workers) as pool:
  list(pool.map(read_one,inputs))
 elapsed=time.perf_counter()-t
 print(f'{label}: {len(inputs)} tiles, {workers} workers, {elapsed:.1f}s, {elapsed/len(inputs):.2f}s/tile',flush=True)
with tempfile.TemporaryDirectory(prefix='musanga_io_') as tmp:
 bench('Drive direct (8 workers)',sample,8)
 t=time.perf_counter()
 with ThreadPoolExecutor(max_workers=8) as pool:
  list(pool.map(lambda p: shutil.copyfile(p,Path(tmp)/p.name),sample))
 print(f'Copy Drive -> local (8 workers): {time.perf_counter()-t:.1f}s',flush=True)
 local=[Path(tmp)/p.name for p in sample]
 bench('Local read (8 workers)',local,8)
print('Diagnostic complete. No existing files changed.')


## 29 — Fast restartable stripe reconstruction (V3_1 strategy)

CPU-only. Reads cached V4 prediction tiles concurrently, validates them, writes 1,024-row stripes on Colab's local disk, uploads completed stripes atomically to Drive, then assembles the final georeferenced probability/binary mosaic with checkpoints. Preserves completed mosaics and leaves legacy partial TIFFs untouched. Requires sufficient temporary local disk space and Google Drive quota; interrupted *local incomplete* stripes must be restarted, but uploaded stripes are reused.


In [ ]:
# NB07 Section 29 — V3_1-style stripe assembly from saved V4 prediction tiles.
# CPU only. Reads tiles in bounded parallel batches; builds locally, uploads
# completed stripes atomically; assembles final mosaic with stripe checkpoints.
# Existing complete mosaics are reused; legacy .partial.tif files are untouched.
from pathlib import Path
from rasterio.transform import Affine
from rasterio.windows import Window
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict
import rasterio,numpy as np,pandas as pd
import hashlib,re,json,os,time,shutil,tempfile,math
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in (
 Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
 Path('/content/drive/MyDrive/Musanga/Final')) if p.is_dir()),None)
assert ROOT is not None,'Musanga Final directory not found'
BASE=ROOT/'Outputs/v4/07_RGB_MS_100m_ROI_complete_sensors_v4'
TILES=BASE/'prediction_tiles'
MERGED=BASE/'reconstructed_mosaics'
MERGED.mkdir(parents=True,exist_ok=True)
MODEL='836ab7d77c1c';POLICY='47890f94c6cf';HALO=64;CORE=256
EXPECTED={'Baego':1568,'Mos_1_2_transparent_mosaic_group1_georef':18464,
'Mos_3_4_transparent_mosaic_group1_georef':13816,
'Mos_5_6_transparent_mosaic_group1_georef':6582,
'Yangambi_georef':4552,'Yoko_kvo_georef':3272}
PATTERN=re.compile(rf'qa{POLICY}_m{MODEL}_halo{HALO:03d}_r(\d+)_c(\d+)\.tif$')
REBUILD=False  # False = preserve/reuse all compatible outputs and stripe checkpoints
STRIPE_TILE_ROWS=4  # 4 x 256 pixels, adapted from V3_1 1024-row stripes
READ_WORKERS=8       # bounded parallel Google Drive reads; lower to 4 if Drive throttles
READ_BATCH=32        # do not preload an entire stripe in RAM
LOG_EVERY=250
LOCAL=Path('/content/musanga_nb07_stripe_scratch')
LOCAL.mkdir(parents=True,exist_ok=True)

def atomic_json(p,data):
 tmp=p.with_name(p.name+'.tmp')
 tmp.write_text(json.dumps(data,indent=2))
 os.replace(tmp,p)
def aligned(a,b):
 return np.allclose(tuple(a)[:6],tuple(b)[:6],rtol=0,atol=1e-6)
def tile_open(entry,crs,origin):
 row,col,path=entry
 with rasterio.open(path) as ds:
  assert ds.count==2 and ds.nodata==-9999 and ds.crs==crs and ds.dtypes==('float32','float32'),path
  assert ds.tags().get('qa_signature')==POLICY and ds.tags().get('edge_halo_px')==str(HALO),path
  assert 0<ds.width<=CORE and 0<ds.height<=CORE,path
  anchor=ds.transform*Affine.translation(-col*CORE,-row*CORE)
  assert aligned(anchor,origin),f'Misaligned tile: {path}'
  a=ds.read()
  valid=a[0]!=-9999
  assert np.isfinite(a).all() and np.array_equal(valid,a[1]!=-9999),path
  assert np.all((a[0][valid]>=0)&(a[0][valid]<=1)),path
  assert np.array_equal(a[1][valid],(a[0][valid]>=.5).astype('float32')),path
  return row,col,a
def stripe_valid(path,spec,first_row,last_row):
 if not path.exists():return False
 try:
  with rasterio.open(path) as ds:
   trans=Affine(*spec['transform'])*Affine.translation(0,(first_row-spec['min_row'])*CORE)
   expected_h=min(spec['height']-(first_row-spec['min_row'])*CORE,(last_row-first_row+1)*CORE)
   return (ds.count==2 and ds.dtypes==('float32','float32') and ds.nodata==-9999
    and ds.width==spec['width'] and ds.height==expected_h and str(ds.crs)==spec['crs']
    and aligned(ds.transform,trans) and ds.tags().get('SPEC')==spec['key']
    and ds.tags().get('COMPLETE')=='YES')
 except Exception:return False
def output_valid(path,spec):
 if not path.exists():return False
 try:
  with rasterio.open(path) as ds:
   return (ds.count==2 and ds.dtypes==('float32','float32') and ds.nodata==-9999
    and ds.width==spec['width'] and ds.height==spec['height']
    and str(ds.crs)==spec['crs'] and aligned(ds.transform,spec['transform']))
 except Exception:return False
def profile(width,height,crs,transform):
 return dict(driver='GTiff',count=2,dtype='float32',width=int(width),height=int(height),
  crs=crs,transform=transform,nodata=-9999,tiled=True,blockxsize=256,blockysize=256,
  compress='deflate',predictor=3,BIGTIFF='YES',SPARSE_OK='NO')
def run(name,count):
 tick=time.monotonic()
 print(f'\n[{name}] Inventory...',flush=True)
 folder=TILES/name
 assert folder.is_dir(),folder
 matched=[];older=0
 for p in folder.glob('*.tif'):
  m=PATTERN.fullmatch(p.name)
  if m:matched.append((int(m.group(1)),int(m.group(2)),p))
  else:older+=1
 assert len(matched)==count,(name,len(matched),count)
 matched.sort(key=lambda x:(x[0],x[1]))
 assert len({(r,c) for r,c,p in matched})==count,'Duplicate positions'
 h=hashlib.sha256()
 for r,c,p in matched:
  st=p.stat()
  h.update(f'{p.name}/{st.st_size}/{st.st_mtime_ns}\n'.encode())
 digest=h.hexdigest()
 full=MERGED/f'{name}_Musanga_probability_binary_v4.tif'
 report=MERGED/f'{name}_reconstruction.json'
 if not REBUILD and full.exists():
  assert report.exists(),f'{name}: existing full TIFF has no report'
  meta=json.loads(report.read_text())
  assert meta.get('digest')==digest and meta.get('model')==MODEL and meta.get('policy')==POLICY
  assert output_valid(full,meta),f'{name}: existing output failed header verification'
  print(f'[ALREADY COMPLETE] {name} in {time.monotonic()-tick:.0f}s',flush=True)
  return meta
 min_r=min(x[0] for x in matched);max_r=max(x[0] for x in matched)
 min_c=min(x[1] for x in matched);max_c=max(x[1] for x in matched)
 # First/last & extreme-edge header samples establish the shared native grid.
 sample_idx=sorted(set([0,count-1]+list(range(0,count,max(1,count//24)))))
 sample={p.name:(r,c,p) for r,c,p in [matched[i] for i in sample_idx]}
 for target in (min_r,max_r,min_c,max_c):
  key=(lambda v: v[0]==target) if target in (min_r,max_r) else (lambda v: v[1]==target)
  for entry in (next(v for v in matched if key(v)),next(v for v in reversed(matched) if key(v))):
   sample[entry[2].name]=entry
 origin=None;crs=None;edge_w=None;edge_h=None
 print(f'[{name}] Preflight {len(sample)} sample headers...',flush=True)
 for row,col,p in sample.values():
  with rasterio.open(p) as ds:
   assert ds.count==2 and ds.nodata==-9999 and ds.crs and ds.dtypes==('float32','float32'),p
   assert ds.tags().get('qa_signature')==POLICY and ds.tags().get('edge_halo_px')==str(HALO),p
   assert 0<ds.width<=CORE and 0<ds.height<=CORE,p
   anchor=ds.transform*Affine.translation(-col*CORE,-row*CORE)
   if origin is None:origin=anchor;crs=ds.crs
   else:assert ds.crs==crs and aligned(anchor,origin),p
   if col==max_c:edge_w=ds.width if edge_w is None else max(edge_w,ds.width)
   if row==max_r:edge_h=ds.height if edge_h is None else max(edge_h,ds.height)
 assert edge_w is not None and edge_h is not None
 width=(max_c-min_c)*CORE+edge_w
 height=(max_r-min_r)*CORE+edge_h
 affine=origin*Affine.translation(min_c*CORE,min_r*CORE)
 spec=dict(name=name,count=count,model=MODEL,policy=POLICY,halo=HALO,digest=digest,
  width=width,height=height,crs=str(crs),transform=list(affine)[:6],
  min_row=min_r,max_row=max_r,min_col=min_c,max_col=max_c)
 # Key excludes runtime-specific concurrency and progress parameters.
 spec['key']=hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()[:24]
 stripe_root=MERGED/'29_stripes_v4'/name/spec['key']
 stripe_root.mkdir(parents=True,exist_ok=True)
 print(f'[{name}] Native map {width:,} x {height:,}; stripe cache {stripe_root}; elapsed {time.monotonic()-tick:.0f}s',flush=True)
 bystripe=defaultdict(list)
 for entry in matched:
  bystripe[(entry[0]-min_r)//STRIPE_TILE_ROWS].append(entry)
 nstripes=(max_r-min_r)//STRIPE_TILE_ROWS+1
 assert len(bystripe)<=nstripes
 n_cached=0;n_new=0
 for i in range(nstripes):
  first=min_r+i*STRIPE_TILE_ROWS
  last=min(max_r,first+STRIPE_TILE_ROWS-1)
  dst=stripe_root/f'stripe_{i:05d}.tif'
  if not REBUILD and stripe_valid(dst,spec,first,last):
   n_cached+=1
   print(f'[{name}] Stripe {i+1}/{nstripes} CACHE HIT',flush=True)
   continue
  assert not dst.exists(),f'Incompatible cached stripe, refusing overwrite: {dst}'
  tasks=bystripe.get(i,[])
  assert tasks,f'{name}: empty stripe {i}, requires explicit handling'
  hh=min(height-(first-min_r)*CORE,(last-first+1)*CORE)
  trans=affine*Affine.translation(0,(first-min_r)*CORE)
  local=LOCAL/f'{name}_{i:05d}.tif'
  local.unlink(missing_ok=True)
  try:
   with rasterio.open(local,'w',**profile(width,hh,crs,trans)) as out:
    out.set_band_description(1,'Musanga probability')
    out.set_band_description(2,'Musanga binary at 0.5')
    # Explicit NoData initialization is necessary; unwritten sparse blocks
    # otherwise read as zero (falsely indicating zero Musanga cover).
    t_init=time.monotonic()
    for yy in range(0,hh,256):
     block=np.full((2,min(256,hh-yy),width),-9999.,dtype='float32')
     out.write(block,window=Window(0,yy,width,block.shape[1]))
    print(f'[{name}] Stripe {i+1}/{nstripes}: NoData initialization {time.monotonic()-t_init:.1f}s',flush=True)
    read_s=0.;write_s=0.
    with ThreadPoolExecutor(max_workers=READ_WORKERS) as pool:
     for start in range(0,len(tasks),READ_BATCH):
      subset=tasks[start:start+READ_BATCH]
      t_read=time.monotonic()
      data_batch=list(pool.map(lambda item:tile_open(item,crs,origin),subset))
      read_s+=time.monotonic()-t_read
      t_write=time.monotonic()
      for row,col,data in data_batch:
       y=(row-first)*CORE;x=(col-min_c)*CORE
       assert 0<=x and 0<=y and x+data.shape[2]<=width and y+data.shape[1]<=hh
       out.write(data,window=Window(x,y,data.shape[2],data.shape[1]))
      write_s+=time.monotonic()-t_write
      del data_batch
      done=start+len(subset)
      if done%128<READ_BATCH or done==len(tasks):
       print(f'[{name}] Stripe {i+1}/{nstripes}: {done:,}/{len(tasks):,} tiles; Drive read+QA {read_s:.1f}s; local writes {write_s:.1f}s; elapsed {time.monotonic()-tick:.0f}s',flush=True)
    out.update_tags(SPEC=spec['key'],COMPLETE='YES',MODEL=MODEL,POLICY=POLICY)
   assert stripe_valid(local,spec,first,last),'Local stripe verification failed'
   pending=dst.with_name(dst.name+'.uploading')
   if pending.exists():pending.unlink()
   shutil.copyfile(local,pending)
   os.replace(pending,dst)
   assert stripe_valid(dst,spec,first,last),'Stripe upload failed QA'
   n_new+=1
   print(f'[{name}] Stripe {i+1}/{nstripes} SAVED; elapsed {time.monotonic()-tick:.0f}s',flush=True)
  finally:
   local.unlink(missing_ok=True)
 # All stripe fingerprints and georeferencing checked; assemble final
 # sequentially using persistent checkpoint to survive interruptions.
 assembling=MERGED/f'{name}_stripe_assembling_v4.tif'
 ckpt=MERGED/f'{name}_stripe_assembly_checkpoint.json'
 for i in range(nstripes):
  first=min_r+i*STRIPE_TILE_ROWS;last=min(max_r,first+STRIPE_TILE_ROWS-1)
  assert stripe_valid(stripe_root/f'stripe_{i:05d}.tif',spec,first,last)
 if not assembling.exists():
  assert not ckpt.exists(),'Assembly checkpoint without raster; inspect manually'
  with rasterio.open(assembling,'w',**profile(width,height,crs,affine)) as ds:
   ds.set_band_description(1,'Musanga probability')
   ds.set_band_description(2,'Musanga binary at 0.5')
   ds.update_tags(SPEC=spec['key'],MODEL=MODEL,POLICY=POLICY)
  atomic_json(ckpt,{'key':spec['key'],'next':0})
 else:
  assert ckpt.exists(),'Assembly raster without checkpoint; inspect manually'
 state=json.loads(ckpt.read_text())
 assert state['key']==spec['key'],'Assembly input fingerprint changed'
 start=int(state['next']);assert 0<=start<=nstripes
 print(f'[{name}] Final assembly from stripe {start+1}/{nstripes}...',flush=True)
 with rasterio.open(assembling,'r+') as out:
  assert output_valid(assembling,spec)
  for i in range(start,nstripes):
   first=min_r+i*STRIPE_TILE_ROWS
   src=stripe_root/f'stripe_{i:05d}.tif'
   with rasterio.open(src) as tile:
    for yy in range(0,tile.height,256):
     hh=min(256,tile.height-yy)
     a=tile.read(window=Window(0,yy,width,hh))
     out.write(a,window=Window(0,(first-min_r)*CORE+yy,width,hh))
   atomic_json(ckpt,{'key':spec['key'],'next':i+1})
   print(f'[{name}] Assembled {i+1}/{nstripes} stripes; elapsed {time.monotonic()-tick:.0f}s',flush=True)
 assert output_valid(assembling,spec)
 assert not full.exists(),f'Will not overwrite existing output: {full}'
 os.replace(assembling,full)
 result={**spec,'status':'complete','output':str(full),
  'excluded_other_policy_tiles':older,'stripes':nstripes,
  'new_stripes':n_new,'reused_stripes':n_cached,'method':'local_stripe_parallel_reads_v4'}
 atomic_json(report,result)
 ckpt.unlink(missing_ok=True)
 print(f'[{name}] COMPLETE in {(time.monotonic()-tick)/60:.1f} minutes',flush=True)
 return result
reports=[run(name,count) for name,count in EXPECTED.items()]
frame=pd.DataFrame(reports)
frame.to_csv(MERGED/'29_reconstruction_summary.csv',index=False)
display(frame[['name','count','width','height','status']])
print('Final mosaics:',MERGED)
